In [ ]:
# 05_langchain_vs_raw/raw_tinyllama.ipynb
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype).to(device)

messages = [
    {"role": "system", "content": "You are a patient tutor."},
    {"role": "user", "content": "Explain overfitting in three sentences."},
]

# 1. Format the messages with the model's own chat template
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# 2. Tokenise
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# 3. Generate (do_sample=False is greedy decoding, the equivalent of temperature=0)
with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=150, do_sample=False)

# 4. Decode only the newly generated tokens (skip the prompt)
new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))